In [1]:
# Task 1: Load Data with textFile

from pyspark import SparkContext, SparkConf

conf = SparkConf().setMaster("local[*]").setAppName("DataIO")
sc = SparkContext.getOrCreate(conf=conf)

"""
Created sample data file 'sales_data.csv':

product_id,name,category,price,quantity
P001,Laptop,Electronics,999.99,5
P002,Mouse,Electronics,29.99,50
P003,Desk,Furniture,199.99,10
P004,Chair,Furniture,149.99,20
P005,Monitor,Electronics,299.99,15
P006,Keyboard,Electronics,79.99,30
P007,Lamp,Furniture,49.99,25
"""

# Load the CSV file
lines = sc.textFile("sales_data.csv")

# Skip header line
header = lines.first()
data = lines.filter(lambda line: line != header)

print(f"Header: {header}")
print(f"Data records: {data.count()}")
print(f"First record: {data.first()}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/30 12:11:23 WARN Utils: Your hostname, AsusLaptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/30 12:11:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/30 12:11:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/30 12:11:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Header: product_id,name,category,price,quantity
Data records: 7
First record: P001,Laptop,Electronics,999.99,5


In [2]:
# Task 2: Parse CSV Records

def parse_record(line):
    """Parse CSV line into structured data."""
    parts = line.split(",")
    return {
        "product_id": parts[0],
        "name": parts[1],
        "category": parts[2],
        "price": float(parts[3]),
        "quantity": int(parts[4])
    }

# Parse all records
parsed = data.map(parse_record)

# Show parsed data
for record in parsed.take(3):
    print(record)


{'product_id': 'P001', 'name': 'Laptop', 'category': 'Electronics', 'price': 999.99, 'quantity': 5}
{'product_id': 'P002', 'name': 'Mouse', 'category': 'Electronics', 'price': 29.99, 'quantity': 50}
{'product_id': 'P003', 'name': 'Desk', 'category': 'Furniture', 'price': 199.99, 'quantity': 10}


In [3]:
# Task 3: Process and Save Results

# Calculate revenue for each product
revenue = parsed.map(lambda r: f"{r['product_id']},{r['name']},{r['price'] * r['quantity']:.2f}")

# Save to output directory
# YOUR CODE: Use saveAsTextFile to save revenue data
revenue.saveAsTextFile("output")

In [4]:
# Task 4: Load Multiple Files

# YOUR CODE: Create sales_data_2.csv with more records
import csv
from prettytable import PrettyTable

# Create new sales_data2.csv
rows = [
    ["product_id", "name", "category", "price", "quantity"],
    ["P008", "Laptop",   "Electronics", 888.88, 4],
    ["P009", "Mouse",   "Electronics", 18.88, 8],
    ["P010", "Desk",     "Furniture",   288.88, 4],
    ["P011", "Chair",    "Furniture",   148.88, 18],
    ["P012", "Monitor",  "Electronics", 288.88, 14],
    ["P013", "Keyboard", "Electronics", 78.88, 38],
    ["P014", "Lamp",     "Furniture",   48.88, 28]
]

# Write to the CSV file
with open("sales_data_2.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

# YOUR CODE: Load all CSV files using wildcard pattern
all_data = sc.textFile("sales_data*.csv")

header = "product_id,name,category,price,quantity"

all_data = all_data.filter(lambda line: line != header)

table = PrettyTable()
table.field_names = ["product_id", "name", "category", "price", "quantity"]

for row in all_data.collect():
    table.add_row(row.split(","))

print(table)


+------------+----------+-------------+--------+----------+
| product_id |   name   |   category  | price  | quantity |
+------------+----------+-------------+--------+----------+
|    P001    |  Laptop  | Electronics | 999.99 |    5     |
|    P002    |  Mouse   | Electronics | 29.99  |    50    |
|    P003    |   Desk   |  Furniture  | 199.99 |    10    |
|    P004    |  Chair   |  Furniture  | 149.99 |    20    |
|    P005    | Monitor  | Electronics | 299.99 |    15    |
|    P006    | Keyboard | Electronics | 79.99  |    30    |
|    P007    |   Lamp   |  Furniture  | 49.99  |    25    |
|    P008    |  Laptop  | Electronics | 888.88 |    4     |
|    P009    |  Mouse   | Electronics | 18.88  |    8     |
|    P010    |   Desk   |  Furniture  | 288.88 |    4     |
|    P011    |  Chair   |  Furniture  | 148.88 |    18    |
|    P012    | Monitor  | Electronics | 288.88 |    14    |
|    P013    | Keyboard | Electronics | 78.88  |    38    |
|    P014    |   Lamp   |  Furniture  | 

In [6]:
# Task 5: Coalesce Output

# YOUR CODE: Use coalesce(1) before saveAsTextFile
# This creates a single output file instead of multiple parts
all_data.coalesce(1).saveAsTextFile("all_sales_data")